# Занятие 2, демо 1. Почему конечную память приходится редактировать

Состояние $S\in\mathbb R^{d_v\times d_k}$ - это вся память. Записей в неё
сколько угодно, а размер фиксирован. Посмотрим, что из этого следует.

In [ ]:
import torch

torch.set_num_threads(1)

"""Конечная память: запись, интерференция, перезапись.

Материал занятия 2. Additive-запись и delta rule - два правила **записи** в
одно и то же состояние. Нормализация устроена иначе: состояние копится тем же
additive-правилом, но рядом копится масса ядра, и меняется способ **чтения**.
"""
import torch


def additive_write(state, k, v):
    """Слагаемое: S <- S + v k^T. Прошлое не трогается."""
    return state + v.unsqueeze(-1) * k.unsqueeze(-2)


def delta_write(state, k, v, beta=1.0):
    """Delta rule: сначала читается то, что уже лежит по ключу, потом правится.

    S <- S + beta * (v - S k) k^T. При beta = 1 и единичном ключе чтение по
    этому ключу становится ровно v.
    """
    current = (state @ k.unsqueeze(-1)).squeeze(-1)
    return state + beta * (v - current).unsqueeze(-1) * k.unsqueeze(-2)


def read(state, q):
    """Чтение: y = S q."""
    return (state @ q.unsqueeze(-1)).squeeze(-1)


def unit(*values, dtype=torch.float64):
    """Единичный вектор заданного направления."""
    t = torch.tensor(values, dtype=dtype)
    return t / t.norm()


def rotate(vector, angle):
    """Поворот двумерного вектора на угол в радианах."""
    c, s = torch.cos(torch.tensor(angle, dtype=vector.dtype)), torch.sin(
        torch.tensor(angle, dtype=vector.dtype))
    return torch.stack([c * vector[0] - s * vector[1],
                        s * vector[0] + c * vector[1]])


def zero_state(d_k, d_v, dtype=torch.float64):
    """Состояние имеет форму d_v x d_k: та же, что на семинаре."""
    return torch.zeros(d_v, d_k, dtype=dtype)


def normalized_additive(keys, values, q, eps=0.0):
    """Нормированное additive linear attention, в той же форме, что и delta.

    Состояние копится обычным additive-правилом, рядом копится масса ядра
    z = sum k_i. Меняется только чтение: y = S q / (z^T q). Возвращает
    (состояние, масса, ответ).

    Так видно, что это не альтернативное правило записи: в S лежит ровно то же,
    что и без нормализации.
    """
    state = zero_state(q.shape[-1], values[0].shape[-1], dtype=values[0].dtype)
    mass = torch.zeros_like(q)
    for k, v in zip(keys, values):
        state = additive_write(state, k, v)
        mass = mass + k
    return state, mass, read(state, q) / (q @ mass + eps)


def delta_sequence(keys, values, q, d_k, d_v, beta=1.0):
    """Последовательная запись delta rule и чтение по q."""
    state = zero_state(d_k, d_v, dtype=values[0].dtype)
    for k, v in zip(keys, values):
        state = delta_write(state, k, v, beta=beta)
    return state, read(state, q)


def random_memory(n, d_k, d_v, generator):
    """n случайных ассоциаций: единичные ключи, гауссовы значения."""
    keys = []
    for _ in range(n):
        key = torch.randn(d_k, generator=generator, dtype=torch.float64)
        keys.append(key / key.norm())
    values = [torch.randn(d_v, generator=generator, dtype=torch.float64)
              for _ in range(n)]
    state = zero_state(d_k, d_v)
    for key, value in zip(keys, values):
        state = additive_write(state, key, value)
    return state, keys, values


def interference_rms(n, d_k, d_v, trials, seed):
    """Среднеквадратичная ошибка чтения первой пары по многим реализациям.

    Одна реализация ничего не показывает: ошибка случайна и от n к n может
    как расти, так и падать. Растёт именно типичная величина.
    """
    generator = torch.Generator().manual_seed(seed)
    total = 0.0
    for _ in range(trials):
        state, keys, values = random_memory(n, d_k, d_v, generator)
        total += float((read(state, keys[0]) - values[0]).norm()) ** 2
    return (total / trials) ** 0.5

## Шаг 1. Две записи по одному ключу

Пишем по ключу $k$ сначала $v_1$, потом $v_2$. Читаем тем же $k$.

Чего мы хотим: память возвращает $v_2$ - последнее, что записали.

In [ ]:
k = unit(1.0, 0.0)
v1 = torch.tensor([1.0, 0.0], dtype=torch.float64)
v2 = torch.tensor([0.0, 1.0], dtype=torch.float64)

s_add = additive_write(additive_write(zero_state(2, 2), k, v1), k, v2)
s_delta = delta_write(delta_write(zero_state(2, 2), k, v1), k, v2)

print("записали v1 =", v1.tolist(), "потом v2 =", v2.tolist())
print()
print("additive-запись S <- S + v k^T :", read(s_add, k).tolist())
print("delta rule                     :", read(s_delta, k).tolist())
print()
print("Additive-запись вернула сумму обоих. Перезаписи не произошло.")

## Шаг 2. Ключи не совпадают, а почти совпадают

Это и есть обычный случай: ключи - выходы сети, и точно они обычно не
совпадают. Пишем $v_1$ по ключу $k_1$, потом $v_2$ по ключу $k_2$, повёрнутому
на угол $\theta$. Читаем по $k_1$.

Величина примеси считается сразу: $y(k_1)=v_1+(k_1^\top k_2)\,v_2$.

In [ ]:
import math

print(f"{'угол':>6}  {'прочитали по k1':>28}  {'чужого в ответе':>16}")
for degrees in (90, 45, 20, 5):
    k2 = rotate(k, degrees * math.pi / 180)
    state = additive_write(additive_write(zero_state(2, 2), k, v1), k2, v2)
    got = read(state, k)
    print(f"{degrees:>5}°  {str([round(x, 4) for x in got.tolist()]):>28}"
          f"  {float(got[1]):>16.4f}")

print()
print("Нужно было [1.0, 0.0]. Вторая координата - целиком чужая запись.")

## Шаг 3. Что при этом делает delta rule

Обе записи оставляют в чтении по $k_1$ одну и ту же чужую примесь - она равна
$c=k_1^\top k_2$ и от правила записи в этом примере не зависит. Различие в
другом: **что читается по тому ключу, по которому писали последним**.

$$
y_{\text{add}}(k_2)=c\,v_1+v_2,\qquad y_{\Delta}(k_2)=v_2
$$

$$
y_{\text{add}}(k_1)=v_1+c\,v_2,\qquad y_{\Delta}(k_1)=(1-c^2)\,v_1+c\,v_2
$$

Delta rule возвращает по $k_2$ ровно то, что записали. Цена видна во второй
строке: коррекция по $k_2$ задела чтение по соседнему $k_1$.

In [ ]:
angle = 45
k2 = rotate(k, angle * math.pi / 180)
s_add = additive_write(additive_write(zero_state(2, 2), k, v1), k2, v2)
s_delta = delta_write(delta_write(zero_state(2, 2), k, v1), k2, v2)

print(f"угол {angle}°, записывали v1 = {v1.tolist()} по k1, затем"
      f" v2 = {v2.tolist()} по k2\n")
print(f"{'':<16} {'читаем по k2':>22} {'читаем по k1':>22}")
for name, state in (("additive-запись", s_add), ("delta rule", s_delta)):
    print(f"{name:<16} {str([round(x, 3) for x in read(state, k2).tolist()]):>22}"
          f" {str([round(x, 3) for x in read(state, k).tolist()]):>22}")

print()
print("По k2 delta вернула записанное точно. По k1 - ослабила старую запись.")
print("Это обмен: коррекция одного перекрывающегося ключа не изолирована")
print("от соседних.")

## Шаг 4. Как растёт интерференция с числом записей

Здесь важно не обмануться: у **одной** памяти ошибка случайна и от $n$ к $n$
может как расти, так и падать. Смотреть надо на типичную величину - корень из
среднего квадрата по многим реализациям.

Для единичных случайных ключей и гауссовых значений она считается точно:

$$
\mathbb E\lVert e\rVert^2=(n-1)\frac{d_v}{d_k}
$$

Проверим. Можно менять seed и число реализаций - картина не изменится.

In [ ]:
d_k = d_v = 8
trials = 300

print(f"{'записей':>8} {'RMS ошибки':>12} {'предсказание':>14}")
for n in (1, 2, 4, 8, 16, 32):
    rms = interference_rms(n, d_k, d_v, trials, seed=100 + n)
    predicted = math.sqrt((n - 1) * d_v / d_k)
    print(f"{n:>8} {rms:>12.3f} {predicted:>14.3f}")

print()
print(f"ключей размерности {d_k}: попарно ортогональных среди них не больше {d_k}.")
print("Дальше перекрытие неизбежно, и оно накапливается.")

## Что из этого следует

Новое слагаемое $vk^\top$ добавляется к состоянию, не удаляя предыдущих; при
чтении они смешиваются пропорционально сходству ключей. Типичная интерференция
растёт с числом записей - у отдельной памяти ошибка при этом может и скакать.

Отсюда и берётся вопрос занятия: **как редактировать конечное состояние**, а не
только дописывать в него. Delta rule - первый ответ: прежде чем писать, прочитай
то, что уже лежит по этому ключу, и запиши разницу.